<a href="https://colab.research.google.com/github/IstaftyKevaYuftika/pm-turi2-preprocessing-IstaftyKevaYuftika/blob/main/Salinan_dari_PM_P4_IstaftyKevaYuftika_2488010007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np

# Dataset karyawan (sengaja mengandung masalah kualitas)
data = {
    'usia'      : [25, 32, np.nan, 45, 28, 51, 38, np.nan, 29, 41],
    'pendapatan': [4_000_000, 7_500_000, 5_200_000, 12_000_000, 4_800_000,
                   15_000_000, 8_100_000, 6_300_000, np.nan, 9_900_000],
    'pendidikan': ['SMA','S1','SMA','S2','SMA','S2','S1','S1','SMA','S2'],  # ordinal
    'kota'      : ['Bandung','Jakarta','Bandung','Surabaya','Jakarta',
                   'Surabaya','Jakarta','Bandung','Bandung','Jakarta'],    # nominal
    'membeli'   : ['Tidak','Ya','Tidak','Ya','Tidak','Ya','Ya','Tidak','Tidak','Ya']  # target
}
df = pd.DataFrame(data)
df

,usia,pendapatan,pendidikan,kota,membeli
0,25.0,4000000.0,SMA,Bandung,Tidak
1,32.0,7500000.0,S1,Jakarta,Ya
2,NaN,5200000.0,SMA,Bandung,Tidak
3,45.0,12000000.0,S2,Surabaya,Ya
4,28.0,4800000.0,SMA,Jakarta,Tidak
5,51.0,15000000.0,S2,Surabaya,Ya
6,38.0,8100000.0,S1,Jakarta,Ya
7,NaN,6300000.0,S1,Bandung,Tidak
8,29.0,NaN,SMA,Bandung,Tidak
9,41.0,9900000.0,S2,Jakarta,Ya


In [11]:
df['usia'] = df['usia'].fillna(df['usia'].median())
df['pendapatan'] = df['pendapatan'].fillna(df['pendapatan'].median())
df

,usia,pendapatan,pendidikan,kota,membeli
0,25.0,4000000.0,SMA,Bandung,Tidak
1,32.0,7500000.0,S1,Jakarta,Ya
2,35.0,5200000.0,SMA,Bandung,Tidak
3,45.0,12000000.0,S2,Surabaya,Ya
4,28.0,4800000.0,SMA,Jakarta,Tidak
5,51.0,15000000.0,S2,Surabaya,Ya
6,38.0,8100000.0,S1,Jakarta,Ya
7,35.0,6300000.0,S1,Bandung,Tidak
8,29.0,7500000.0,SMA,Bandung,Tidak
9,41.0,9900000.0,S2,Jakarta,Ya


In [12]:
X = df.drop(columns=['membeli'])
y = df['membeli']
print("Bentuk X:", X.shape, "| Bentuk y:", y.shape)

Bentuk X: (10, 4) | Bentuk y: (10,)


In [13]:
X['pendidikan'] = X['pendidikan'].map({'SMA':0,'S1':1,'S2':2}) # ordinal
X = pd.get_dummies(X, columns=['kota'], dtype=int) # nominal
y = y.map({'Tidak':0,'Ya':1})
X

,usia,pendapatan,pendidikan,kota_Bandung,kota_Jakarta,kota_Surabaya
0,25.0,4000000.0,0,1,0,0
1,32.0,7500000.0,1,0,1,0
2,35.0,5200000.0,0,1,0,0
3,45.0,12000000.0,2,0,0,1
4,28.0,4800000.0,0,0,1,0
5,51.0,15000000.0,2,0,0,1
6,38.0,8100000.0,1,0,1,0
7,35.0,6300000.0,1,1,0,0
8,29.0,7500000.0,0,1,0,0
9,41.0,9900000.0,2,0,1,0


In [14]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
 X, y, test_size=0.3, random_state=42)
# Salinan sebelum scaling
X_train_mentah, X_test_mentah = X_train.copy(), X_test.copy()

print("Latih:", X_train.shape, "| Uji:", X_test.shape)


Latih: (7, 6) | Uji: (3, 6)


In [15]:
from sklearn.preprocessing import StandardScaler
num = ['usia','pendapatan','pendidikan']
sc = StandardScaler()
X_train[num] = sc.fit_transform(X_train[num])
X_test[num] = sc.transform(X_test[num]) # cegah data leakage
X_train


,usia,pendapatan,pendidikan,kota_Bandung,kota_Jakarta,kota_Surabaya
0,-1.588665,-1.169416,-1.028992,1,0,0
7,-0.044130,-0.325129,0.171499,1,0,0
2,-0.044130,-0.728919,-1.028992,1,0,0
9,0.882592,0.996363,1.371989,0,1,0
4,-1.125304,-0.875751,-1.028992,0,1,0
3,1.500406,1.767234,1.371989,0,0,1
6,0.419231,0.335617,0.171499,0,1,0


In [16]:
print("Nilai hilang (latih):", X_train.isna().sum().sum())
print("Nilai hilang (uji)  :", X_test.isna().sum().sum())
print()
print("Tipe data:")
print(X_train.dtypes)
print()
print("Rata-rata & std (populasi, ddof=0 seperti StandardScaler) pada data latih (harapan: 0 dan 1):")
print(X_train[num].agg(['mean', lambda c: c.std(ddof=0)]).rename(index={'<lambda>':'std (ddof=0)'}).round(3))

Nilai hilang (latih): 0
Nilai hilang (uji)  : 0

Tipe data:
usia             float64
pendapatan       float64
pendidikan       float64
kota_Bandung       int64
kota_Jakarta       int64
kota_Surabaya      int64
dtype: object

Rata-rata & std (populasi, ddof=0 seperti StandardScaler) pada data latih (harapan: 0 dan 1):
              usia  pendapatan  pendidikan
mean           0.0         0.0        -0.0
std (ddof=0)   1.0         1.0         1.0


**Latihan Mandiri**

Latihan-1

In [17]:
df1 = pd.DataFrame(data)                                # data asli (masih ada NaN)
df1['usia'] = df1['usia'].fillna(df1['usia'].mean())    # isi dengan mean

print("Mean  :", pd.DataFrame(data)['usia'].mean())     # 36.125
print("Median:", pd.DataFrame(data)['usia'].median())   # 35.0
print(df1['usia'].tolist())

Mean  : 36.125
Median: 35.0
[25.0, 32.0, 36.125, 45.0, 28.0, 51.0, 38.0, 36.125, 29.0, 41.0]


Latihan-2

In [18]:
from sklearn.preprocessing import MinMaxScaler

# ambil data latih/uji yang belum di-scaling
Xtr, Xte, _, _ = train_test_split(X, y, test_size=0.3, random_state=42)

mm = MinMaxScaler()
Xtr[num] = mm.fit_transform(Xtr[num])
Xte[num] = mm.transform(Xte[num])

print(Xtr[num].describe())

           usia  pendapatan  pendidikan
count  7.000000    7.000000    7.000000
mean   0.514286    0.398214    0.428571
std    0.349660    0.367808    0.449868
min    0.000000    0.000000    0.000000
25%    0.325000    0.125000    0.000000
50%    0.500000    0.287500    0.500000
75%    0.725000    0.625000    0.750000
max    1.000000    1.000000    1.000000


Latihan-3

In [19]:
df3 = pd.DataFrame(data)
df3['jenis_kelamin'] = ['Pria','Wanita','Pria','Pria','Wanita',
                        'Pria','Wanita','Wanita','Pria','Wanita']

df3 = pd.get_dummies(df3, columns=['jenis_kelamin'], dtype=int)
df3

,usia,pendapatan,pendidikan,kota,membeli,jenis_kelamin_Pria,jenis_kelamin_Wanita
0,25.0,4000000.0,SMA,Bandung,Tidak,1,0
1,32.0,7500000.0,S1,Jakarta,Ya,0,1
2,NaN,5200000.0,SMA,Bandung,Tidak,1,0
3,45.0,12000000.0,S2,Surabaya,Ya,1,0
4,28.0,4800000.0,SMA,Jakarta,Tidak,0,1
5,51.0,15000000.0,S2,Surabaya,Ya,1,0
6,38.0,8100000.0,S1,Jakarta,Ya,0,1
7,NaN,6300000.0,S1,Bandung,Tidak,0,1
8,29.0,NaN,SMA,Bandung,Tidak,1,0
9,41.0,9900000.0,S2,Jakarta,Ya,0,1


Latihan-4

fit_transform pada data latih karena scaler perlu belajar parameter (mean dan std) dari data latih lalu langsung menerapkannya. Data uji hanya di-transform memakai parameter dari data latih, karena data uji dianggap data baru yang belum pernah dilihat model. Jika fit dilakukan juga pada data uji, informasi data uji bocor ke proses latih (data leakage) dan hasil evaluasi jadi terlalu optimistis.

**Refleksi**
1. Karena scaling memakai statistik data (mean, std, min, max). Jika scaling dilakukan sebelum split, statistik data uji ikut memengaruhi nilai data latih, sehingga
informasi dari data uji bocor ke pelatihan (*data leakage*). Akibatnya performa yang terukur pada data uji lebih baik daripada performa sebenarnya di data baru.
Dengan split dulu, scaler hanya "melihat" data latih, dan data uji diperlakukan seperti data yang benar-benar baru.

2. Kapan label encoding dan kapan one-hot encoding?

- **Label/ordinal encoding:** untuk kategori yang punya **urutan bermakna** (mis. SMA < S1 < S2), agar urutan tersebut terjaga sebagai angka 0, 1,

- **One-hot encoding:** untuk kategori **nominal tanpa urutan** (mis. kota). Jika diberi label 0, 1, 2, model bisa keliru mengira Surabaya "lebih besar" daripada Bandung. One-hot menghindari asumsi urutan palsu, dengan konsekuensi jumlah kolom bertambah.

3. | | Normalisasi (Min-Max) | Standardisasi (Z-score) |
|---|---|---|
| Rumus | (x − min) / (max − min) | (x − mean) / std |
| Hasil | Rentang tetap 0–1 | Mean 0, std 1 (tanpa batas rentang) |
| Sensitivitas outlier | Tinggi (min/max terpengaruh) | Lebih rendah |
| Cocok untuk | Data tanpa outlier ekstrem, jaringan saraf, KNN | Data mendekati normal, SVM, regresi logistik, PCA |